In [1]:
# ============================================================
# IMPORTS
# ============================================================
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
import csv, time, json, os


# ============================================================
# CONFIG
# ============================================================
CONFIG_PATH = "ConfigWBBR.json"
INPUT_CSV  = "entrada_referencias.csv"
OUTPUT_CSV = "salida_sucesores_mejorado.csv"
DELAY = 1.5  # Aumentado para mejor estabilidad

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    cfg = json.load(f)

USER = cfg["USER"]
PWD  = cfg["PWD"]
LOGIN_URL = cfg["LOGIN_URL"]
CSPS_URL  = cfg["CSPS_URL"]


# ============================================================
# BROWSER
# ============================================================
opts = Options()
opts.add_argument("--start-maximized")
driver = webdriver.Chrome(options=opts)
W = WebDriverWait(driver, 25)


# ============================================================
# UTILS
# ============================================================
def ensure_detail():
    driver.switch_to.default_content()
    W.until(EC.frame_to_be_available_and_switch_to_it((By.NAME, "detail")))

def open_consulta():
    driver.switch_to.default_content()
    W.until(EC.frame_to_be_available_and_switch_to_it((By.NAME, "menu")))
    driver.find_element(By.ID, "folderIcon9").click()
    driver.find_element(By.XPATH, "//a[@title='Consulta General de Referencia']").click()
    ensure_detail()

def get_todos_sucesores():
    """
    Extrae TODOS los sucesores de la tabla, no solo 2.
    Retorna una cadena con los sucesores separados por coma.
    """
    sucesores = []
    
    try:
        # Esperar a que exista la tabla
        W.until(EC.presence_of_element_located((By.XPATH, "//table[@id='table1']")))
        
        # Dar tiempo adicional para que carguen las filas
        time.sleep(0.8)
        
        # Buscar todas las filas (sin esperar forzosamente que existan)
        rows = driver.find_elements(
            By.XPATH,
            "//table[@id='table1']//tr[position()>1 and td]"
        )
        
        if not rows:
            print("    → No hay filas en la tabla (sin sucesores)")
            return ""
        
        # Extraer TODOS los sucesores de la columna 3
        for r in rows:
            try:
                tds = r.find_elements(By.TAG_NAME, "td")
                if len(tds) >= 3:
                    val = tds[2].text.replace("\xa0", "").strip()
                    if val:
                        sucesores.append(val)
                        print(f"    → Sucesor encontrado: {val}")
            except Exception as e:
                print(f"    → Error extrayendo fila: {e}")
                continue
        
    except TimeoutException:
        print("    → Timeout esperando tabla")
        return ""
    except Exception as e:
        print(f"    → Error general: {e}")
        return ""
    
    # Unir todos los sucesores con coma
    resultado = ", ".join(sucesores)
    print(f"    → Total sucesores extraídos: {len(sucesores)}")
    return resultado


# ============================================================
# LOGIN
# ============================================================
print("Iniciando sesión...")
driver.get(LOGIN_URL)
W.until(EC.presence_of_element_located((By.ID, "userID"))).send_keys(USER)
driver.find_element(By.ID, "password").send_keys(PWD)
driver.find_element(By.ID, "btn-submit").click()

W.until(EC.url_contains("my.dlrportal.com"))
print("✓ Sesión iniciada")

driver.switch_to.new_window("tab")
driver.get(CSPS_URL)
W.until(EC.presence_of_element_located((By.TAG_NAME, "frameset")))
print("✓ Portal CSPS cargado")

open_consulta()
print("✓ Consulta General abierta\n")


# ============================================================
# CSV INPUT
# ============================================================
with open(INPUT_CSV, "r", encoding="utf-8-sig") as f:
    reader = csv.DictReader(f)
    referencias = [r["referencia"].strip() for r in reader if r["referencia"].strip()]

print(f"Total referencias a procesar: {len(referencias)}\n")
print("="*60)


# ============================================================
# CSV OUTPUT
# ============================================================
with open(OUTPUT_CSV, "w", newline="", encoding="utf-8-sig") as f_out:
    writer = csv.writer(f_out)
    writer.writerow(["Referencia", "Sucesores"])  # Solo 2 columnas

    for i, ref in enumerate(referencias, 1):
        print(f"\n[{i}/{len(referencias)}] Procesando: {ref}")
        try:
            ensure_detail()
            inp = W.until(EC.presence_of_element_located((By.ID, "partNr")))
            inp.clear()
            inp.send_keys(ref)
            inp.send_keys(Keys.ENTER)

            # Esperar que aparezca la sección de Sustituciones
            W.until(EC.presence_of_element_located((
                By.XPATH, "//th[contains(.,'Sustituciones')]"
            )))
            
            # Dar tiempo para que cargue el contenido
            time.sleep(0.5)

            sucesores_str = get_todos_sucesores()
            writer.writerow([ref, sucesores_str])
            
            if sucesores_str:
                print(f"[{i}/{len(referencias)}] ✓ OK {ref} → {sucesores_str}")
            else:
                print(f"[{i}/{len(referencias)}] ✓ OK {ref} → (sin sucesores)")

        except Exception as e:
            writer.writerow([ref, ""])
            print(f"[{i}/{len(referencias)}] ✗ ERROR {ref}: {str(e)}")

        time.sleep(DELAY)


# ============================================================
# END
# ============================================================
print("\n" + "="*60)
print("PROCESO COMPLETADO")
print(f"Archivo generado: {OUTPUT_CSV}")
print("="*60)
input("Presiona ENTER para cerrar el navegador...")
driver.quit()

Iniciando sesión...
✓ Sesión iniciada
✓ Portal CSPS cargado
✓ Consulta General abierta

Total referencias a procesar: 101


[1/101] Procesando: 135294A1
    → Sucesor encontrado: 47953510
    → Total sucesores extraídos: 1
[1/101] ✓ OK 135294A1 → 47953510

[2/101] Procesando: 602958R91
    → Sucesor encontrado: 602957R91
    → Total sucesores extraídos: 1
[2/101] ✓ OK 602958R91 → 602957R91

[3/101] Procesando: 87662557
    → Total sucesores extraídos: 0
[3/101] ✓ OK 87662557 → (sin sucesores)

[4/101] Procesando: 84310960
    → No hay filas en la tabla (sin sucesores)
[4/101] ✓ OK 84310960 → (sin sucesores)

[5/101] Procesando: 5801574786
    → No hay filas en la tabla (sin sucesores)
[5/101] ✓ OK 5801574786 → (sin sucesores)

[6/101] Procesando: 9833222
    → Total sucesores extraídos: 0
[6/101] ✓ OK 9833222 → (sin sucesores)

[7/101] Procesando: 1347351C2
    → Sucesor encontrado: 47946917
    → Total sucesores extraídos: 1
[7/101] ✓ OK 1347351C2 → 47946917

[8/101] Procesando: 58018